# Week 3 · Day 7: Test and astronomy reflection

**Course:** W3 synthesis

The teaching question is **“Is this source's absolute G magnitude below 4?”** Class 1 means yes. This is a brightness label, not a verified stellar type. Only `bp_rp` (color) enters the model; magnitude and parallax are used to construct the answer and must stay out of the inputs.

Day 1 saves a source-disjoint 60/20/20 split. Days 2–6 explore training and validation; Day 7 evaluates the reserved test set. Absolute magnitudes are uncorrected for dust, and the archive sample may not represent all Gaia sources.

## Setup

[Prerequisites and setup explanation](week3_prerequisites.ipynb). Run cells from top to bottom. Run Day 1 first. This notebook reads its saved data.

In [ ]:
from pathlib import Path
import sys
import importlib
import numpy as np
import pandas as pd
from IPython.display import display

# Jupyter may start in the project folder or in its notebooks folder.
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'src/classification.py').is_file():
    raise RuntimeError('Open Jupyter from gaia-explorer or gaia-explorer/notebooks.')
sys.path.insert(0, str(ROOT))
from src import classification as data, visualize
# Refresh the data loader if this kernel imported an older version.
data = importlib.reload(data)
visualize = importlib.reload(visualize)  # Pick up updated plot labels and layout.

DATA_PATH = None  # None uses the shared W2 cache; a missing cache is downloaded.
OUTPUT = ROOT / 'data/processed/week3'
OUTPUT.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 20)


## Load the Day 1 data

These lines select the saved splits. `X` contains color and `y` contains the brightness label.

In [ ]:
prepared_path = OUTPUT / 'learning_data.csv'
if not prepared_path.is_file():
    raise FileNotFoundError('Run Week 3 Day 1 first to create learning_data.csv.')

# Keep source IDs as text so their digits are not rounded.
gaia = pd.read_csv(prepared_path, dtype={'source_id': 'string'})
train = gaia.loc[gaia['split'] == 'train'].copy()
validation = gaia.loc[gaia['split'] == 'validation'].copy()
test = gaia.loc[gaia['split'] == 'test'].copy()
feature_columns = ['bp_rp']
X_train = train[feature_columns]
y_train = train['is_bright']
X_val = validation[feature_columns]
y_val = validation['is_bright']
print('Training rows:', len(train), '| Validation rows:', len(validation))


## 1. Read the saved Day 6 choice

Expected output: the fixed model configuration. Run Day 6 first. The fingerprint check prevents accidentally using settings selected for an older version of Day 1 data.

In [ ]:
import hashlib
settings_path = OUTPUT / 'experiment_settings.csv'
if not settings_path.is_file():
    raise FileNotFoundError('Run Week 3 Day 6 first to choose and save the model settings.')
settings = pd.read_csv(settings_path)
choice = settings.iloc[0]
if choice['data_sha256'] != hashlib.sha256(prepared_path.read_bytes()).hexdigest():
    raise ValueError('Day 1 data has changed. Rerun Day 6 before evaluating the test set.')
display(settings)

## 2. Fit that configuration and predict the test set

Keep the same training-only fit and threshold. No choices are made using test results.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression

model = make_pipeline(
    PolynomialFeatures(degree=int(choice['degree']), include_bias=False),
    StandardScaler(),
    LogisticRegression(C=float(choice['C']), max_iter=5000, tol=1e-9),
)
model.fit(X_train, y_train)
y_test = test['is_bright']
test_p = model.predict_proba(test[feature_columns])[:, 1]
threshold = float(choice['threshold'])
test_predicted = (test_p >= threshold).astype(int)

## 3. Report final metrics

Expected outputs: held-out metrics, constant baseline, confusion matrix, precision-recall curve. These are results for this sample. Do not revise the model after looking at them and reuse this test set as a fresh evaluation.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, log_loss
metrics = pd.Series({
    'accuracy': accuracy_score(y_test, test_predicted),
    'precision': precision_score(y_test, test_predicted, zero_division=0),
    'recall': recall_score(y_test, test_predicted, zero_division=0),
    'f1': f1_score(y_test, test_predicted, zero_division=0),
    'log_loss': log_loss(y_test, test_p),
}, name='selected_model')
baseline_p = np.full(len(y_test), y_train.mean())
display(metrics.to_frame())
print('Constant baseline accuracy:', accuracy_score(y_test, baseline_p >= 0.5))
print('Constant baseline log loss:', log_loss(y_test, baseline_p))
visualize.plot_binary_evaluation(y_test, test_p, threshold=threshold)

## Understand the test results using star counts

Class **1** means `abs_g_mag < 4`; class **0** means `abs_g_mag ≥ 4`. The metrics are easier to understand when we count the four possible outcomes. The cell below calculates these counts from the **current test run**, so the explanation still works if the data changes.

For the 486-star result shown in this project: **5** class 1 stars were found, **78** class 1 stars were missed, **0** class 0 stars were wrongly called class 1, and **403** class 0 stars were correctly identified. That gives precision `5 / (5 + 0) = 1.00` and recall `5 / (5 + 78) ≈ 0.06`. The F1 score is low because finding only 5 of 83 real class 1 stars is poor recall, even though every positive prediction was correct.

Accuracy is `(5 + 403) / 486 ≈ 0.840`. A **constant baseline** ignores color and gives every star the training set's class 1 frequency as its probability; at the 0.5 threshold it predicts class 0 for all stars in this example. Since 403 of 486 test stars are class 0, the baseline already reaches `403 / 486 ≈ 0.829` accuracy. The model improves accuracy by only about **1 percentage point**.

The model's log loss `0.391` is lower than the baseline's `0.457`, so its **probabilities** are more informative than giving every star the same probability. At the fixed 0.5 threshold, it still misses most class 1 stars. These results evaluate the teaching brightness label on this sample, not physical stellar types. Do not change the model or threshold based on these test results and treat the same test rows as a fresh final check.

In [ ]:
actual = y_test.to_numpy()
predicted = np.asarray(test_predicted)
true_positive = int(np.sum((actual == 1) & (predicted == 1)))
false_negative = int(np.sum((actual == 1) & (predicted == 0)))
false_positive = int(np.sum((actual == 0) & (predicted == 1)))
true_negative = int(np.sum((actual == 0) & (predicted == 0)))

display(pd.DataFrame({
    'outcome': ['Found class 1', 'Missed class 1', 'False class 1 alarm', 'Correct class 0'],
    'stars': [true_positive, false_negative, false_positive, true_negative],
}))
print(f'Found {true_positive} of {true_positive + false_negative} actual class 1 stars.')
print(f'Accuracy: {metrics["accuracy"]:.3f}; constant baseline: {accuracy_score(y_test, baseline_p >= 0.5):.3f}')
print(f'Log loss: {metrics["log_loss"]:.3f}; constant baseline: {log_loss(y_test, baseline_p):.3f}')

## 4. Inspect mistakes

Expected outputs: confidently wrong examples and probability colors on the HR diagram. Absolute magnitude is used only to interpret the plot, never as an input to the model.

In [ ]:
predictions = test[['source_id', 'bp_rp', 'abs_g_mag', 'is_bright', 'parallax_snr', 'distance_to_label_cut_mag']].copy()
predictions['probability_bright'] = test_p
predictions['predicted_label'] = test_predicted
predictions['correct'] = test_predicted == y_test
predictions['probability_of_true_label'] = np.where(y_test == 1, test_p, 1 - test_p)
display(predictions.loc[~predictions['correct']].sort_values('probability_of_true_label').head(10))
visualize.plot_classification_hr(test, values=test_p, title='Test sample: predicted brightness probability')

## 5. Save and explain

Expected outputs: test predictions and metric CSVs. They replace only the corresponding W3 result files.

In [ ]:
predictions.to_csv(OUTPUT / 'test_predictions.csv', index=False)
metrics.to_csv(OUTPUT / 'test_metrics.csv')
print('Saved results in:', OUTPUT)

## 6. Astronomy summary from the measured test results

These answers describe the saved **486-star test sample**. If you rerun Day 1 with different Gaia data, use the new Day 7 outputs and update the numbers.

- **Compared with the constant baseline, the model** raised accuracy from `0.829` to `0.840`: it correctly classified **408** stars instead of **403**. Its log loss improved from `0.457` to `0.391`, so its probabilities contain more useful information than giving every star the same class 1 probability. The gain in yes/no accuracy is small—about **1 percentage point**—because the baseline already predicts the common class 0 for everyone.
- **Precision `1.00` and recall `0.060` mean** every star the model predicted as class 1 was actually class 1 (**5 correct, 0 false alarms**), but it found only **5 of the 83** actual class 1 stars and missed **78**. This explains the low F1 score of `0.114`.
- **The biggest mistakes appear among genuinely bright stars with relatively redder BP-RP colors.** The most confidently missed examples have BP-RP around **1.6–1.9** yet absolute G magnitudes well **below 4**; the model gives them a very low chance of class 1. Only **14 of the 78** missed class 1 stars are within 0.2 magnitude of the cutoff, so the failures are not limited to borderline labels.
- **Color alone is insufficient because** stars with similar BP-RP colors can have very different absolute magnitudes. The model has one input, `bp_rp`, so it cannot distinguish two stars with the same color but different brightness labels. This exercise does not identify stellar types.
- **Dust, parallax errors, and the chosen magnitude cut affect the interpretation.** Dust can change observed color and apparent brightness. Parallax uncertainty affects the distance and calculated absolute magnitude. The cut at `M_G = 4` is an educational choice; a star near that cut could switch labels if its estimated magnitude changed. These effects are possible limitations, not proven causes of the particular mistakes above.

An unordered TOP-N archive sample is not guaranteed representative of the sky. These magnitudes have no extinction or parallax zero-point correction. Future stellar-type classification needs independent reference labels and a separate evaluation design.


## Reflection questions and answers

1. **Did we identify stellar types?** No. Our target is a constructed brightness label: `is_bright = 1` when estimated absolute G magnitude is below 4. A red giant, hot main-sequence star, or another luminous source could satisfy that rule. We did not train or validate against independently confirmed stellar-type labels, so the model cannot be described as a stellar-type classifier.

2. **What if test performance is weak?** Report it honestly. Here the fixed 0.5 threshold missed 78 of 83 actual class 1 stars, despite making no false class 1 predictions. We can study these errors and design a better future experiment, perhaps with more informative inputs or a threshold chosen on validation data. If we revise the model after seeing these test results, the same test rows no longer provide a fresh final check; we would need another independent test set for a new final evaluation.

3. **How did W2 help?** W2 taught us to calculate absolute magnitude, select inputs without giving away the target, scale features using training data, keep evaluation rows separate, compute predictions with vectorized NumPy, and update weights with gradient descent. W3 reused those habits while changing the output from a continuous magnitude to a 0/1 label and changing the cost from squared error to logistic loss.


## References

Course labs: `../ML-coursework/W3/Optional Labs/` in the workspace.

[ESA Gaia DR3](https://www.cosmos.esa.int/web/gaia/dr3) · [Extinction guidance](https://www.cosmos.esa.int/web/gaia/edr3-extinction-law) · [Scikit-learn logistic regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)